2. Projektni Zadatak: Razvoj Klasifikatora za Analizu
Sentimenta u PySpark-u

In [56]:
from pyspark.sql import SparkSession

sc = SparkSession.builder.appName("FinancialSentimentAnalysis").getOrCreate()


In [57]:
data = sc.read.csv("data.csv", header=True, inferSchema=True)
data.show()

+--------------------+---------+
|            Sentence|Sentiment|
+--------------------+---------+
|The GeoSolutions ...| positive|
|$ESI on lows, dow...| negative|
|For the last quar...| positive|
|According to the ...|  neutral|
|The Swedish buyou...|  neutral|
|$SPY wouldn't be ...| positive|
|Shell's $70 Billi...| negative|
|SSH COMMUNICATION...| negative|
|Kone 's net sales...| positive|
|The Stockmann dep...|  neutral|
|Circulation reven...| positive|
|$SAP Q1 disappoin...| negative|
|The subdivision m...| positive|
|Viking Line has c...|  neutral|
|Ahlstrom Corporat...|  neutral|
|$FB gone green on...| positive|
|$MSFT SQL Server ...| positive|
|According to L+ñn...|  neutral|
|The company 's sh...|  neutral|
|Elcoteq SE is lis...|  neutral|
+--------------------+---------+
only showing top 20 rows


In [58]:
data.printSchema()

root
 |-- Sentence: string (nullable = true)
 |-- Sentiment: string (nullable = true)



In [59]:
from pyspark.sql.functions import count, when, col

data.select([count(when(col(c).isNull(), c)).alias(c) for c in data.columns]).show()

+--------+---------+
|Sentence|Sentiment|
+--------+---------+
|       0|        0|
+--------+---------+



In [60]:
data.groupBy("sentiment").count().show()

+--------------------+-----+
|           sentiment|count|
+--------------------+-----+
|            positive| 1852|
|             neutral| 3130|
|            negative|  859|
| the damage is do...|    1|
+--------------------+-----+



In [61]:
valid_labels = ["positive", "neutral", "negative"]

data = data.filter(col("Sentiment").isin(valid_labels))

data.groupBy("Sentiment").count().show()

+---------+-----+
|Sentiment|count|
+---------+-----+
| positive| 1852|
|  neutral| 3130|
| negative|  859|
+---------+-----+



In [62]:
from pyspark.ml.feature import StringIndexer

label_indexer = StringIndexer(
    inputCol="Sentiment",
    outputCol="label"
)

data = label_indexer.fit(data).transform(data)
data.select("Sentiment", "label").show(10)

+---------+-----+
|Sentiment|label|
+---------+-----+
| positive|  1.0|
| negative|  2.0|
| positive|  1.0|
|  neutral|  0.0|
|  neutral|  0.0|
| positive|  1.0|
| negative|  2.0|
| negative|  2.0|
| positive|  1.0|
|  neutral|  0.0|
+---------+-----+
only showing top 10 rows


In [63]:
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)

print("Train:", train_data.count())
print("Test:", test_data.count())

Train: 4724
Test: 1117


In [64]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover

tokenizer = Tokenizer(
    inputCol="Sentence",
    outputCol="tokens"
)

stop_remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="filtered_tokens"
)

In [65]:
from nltk.stem import PorterStemmer
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, StringType

stemmer = PorterStemmer()

def stem_tokens(tokens):
    return [stemmer.stem(token) for token in tokens]

stem_udf = udf(stem_tokens, ArrayType(StringType()))

In [66]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

def evaluate_model(predictions):
    metrics = {}
    
    for metric in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]:
        evaluator = MulticlassClassificationEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName=metric
        )
        metrics[metric] = evaluator.evaluate(predictions)
    
    return metrics

In [67]:
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier

def get_model(model_name, params=None):
    params = params or {}

    if model_name == "lr":
        return LogisticRegression(featuresCol="features", labelCol="label", **params)
    
    elif model_name == "dt":
        return DecisionTreeClassifier(featuresCol="features", labelCol="label", seed=42, **params)
    
    elif model_name == "rf":
        return RandomForestClassifier(featuresCol="features", labelCol="label", seed=42, **params)
    
    else:
        raise ValueError("Unknown model")

In [68]:
def preprocess_data(df):
    tokenized = tokenizer.transform(df)
    filtered = stop_remover.transform(tokenized)

    stemmed = filtered.withColumn(
        "stemmed_tokens",
        stem_udf(col("filtered_tokens"))
    )

    return stemmed

In [69]:
from pyspark.ml.feature import CountVectorizer, IDF

def build_tfidf_features(train_df, test_df):
    cv = CountVectorizer(
        inputCol="stemmed_tokens",
        outputCol="raw_features"
    )
    cv_model = cv.fit(train_df)

    train_featurized = cv_model.transform(train_df)
    test_featurized = cv_model.transform(test_df)

    idf = IDF(
        inputCol="raw_features",
        outputCol="features"
    )
    idf_model = idf.fit(train_featurized)

    train_final = idf_model.transform(train_featurized)
    test_final = idf_model.transform(test_featurized)

    return train_final, test_final

In [70]:
from pyspark.ml.feature import NGram

def build_bigram_features(train_df, test_df):
    ngram = NGram(
        n=2,
        inputCol="stemmed_tokens",
        outputCol="bigrams"
    )

    train_ngram = ngram.transform(train_df)
    test_ngram = ngram.transform(test_df)

    cv = CountVectorizer(
        inputCol="bigrams",
        outputCol="raw_features"
    )
    cv_model = cv.fit(train_ngram)

    train_featurized = cv_model.transform(train_ngram)
    test_featurized = cv_model.transform(test_ngram)

    idf = IDF(
        inputCol="raw_features",
        outputCol="features"
    )
    idf_model = idf.fit(train_featurized)

    train_final = idf_model.transform(train_featurized)
    test_final = idf_model.transform(test_featurized)

    return train_final, test_final

In [71]:
from pyspark.ml.feature import Word2Vec

def build_word2vec_features(train_df, test_df):
    word2vec = Word2Vec(
        vectorSize=100,
        minCount=1,
        inputCol="stemmed_tokens",
        outputCol="features"
    )

    w2v_model = word2vec.fit(train_df)

    train_final = w2v_model.transform(train_df)
    test_final = w2v_model.transform(test_df)

    return train_final, test_final

In [72]:
def prepare_features(train_data, test_data, feature_method):
    train_processed = preprocess_data(train_data)
    test_processed = preprocess_data(test_data)

    if feature_method == "tfidf":
        return build_tfidf_features(train_processed, test_processed)

    elif feature_method == "bigram_tfidf":
        return build_bigram_features(train_processed, test_processed)

    elif feature_method == "word2vec":
        return build_word2vec_features(train_processed, test_processed)

    else:
        raise ValueError("Unknown feature method")

In [73]:
def train_and_evaluate(train_final, test_final, model_name, params=None):
    model = get_model(model_name, params)
    trained_model = model.fit(train_final)

    predictions = trained_model.transform(test_final)

    metrics = evaluate_model(predictions)

    return {
        "model": trained_model,
        "metrics": metrics
    }

In [74]:
def run_experiment(train_data, test_data, feature_method, model_name, params=None):
    train_final, test_final = prepare_features(train_data, test_data, feature_method)
    result = train_and_evaluate(train_final, test_final, model_name, params)

    return {
        "feature_method": feature_method,
        "model_name": model_name,
        "params": params or {},
        "metrics": result["metrics"],
        "model": result["model"]
    }

In [75]:
param_grid = {
    "lr": [
        {"maxIter": 20},
        {"maxIter": 50}
    ],
    "dt": [
        {"maxDepth": 5},
        {"maxDepth": 10}
    ],
    "rf": [
        {"numTrees": 20},
        {"numTrees": 50}
    ]
}

In [76]:
def run_all_experiments(train_data, test_data, param_grid):
    results = {}

    feature_methods = ["tfidf", "bigram_tfidf", "word2vec"]
    prepared_data = {}

    for feature_method in feature_methods:
        print(f"Preparing {feature_method} features...")
        train_final, test_final = prepare_features(train_data, test_data, feature_method)
        # cache
        train_final = train_final.cache()
        test_final = test_final.cache()
        # force computation
        train_final.count()
        test_final.count()
        prepared_data[feature_method] = (train_final, test_final)

    for feature_method, (train_final, test_final) in prepared_data.items():
        for model_name, param_list in param_grid.items():
            for params in param_list:
                key = f"{feature_method}_{model_name}_{params}"

                print(f"Running {key}...")

                result = train_and_evaluate(
                    train_final,
                    test_final,
                    model_name,
                    params
                )

                results[key] = result["metrics"]

    return results

In [77]:
all_results = run_all_experiments(train_data, test_data, param_grid)

Preparing tfidf features...
Preparing bigram_tfidf features...
Preparing word2vec features...
Running tfidf_lr_{'maxIter': 20}...
Running tfidf_lr_{'maxIter': 50}...
Running tfidf_dt_{'maxDepth': 5}...


Running tfidf_dt_{'maxDepth': 10}...


Running tfidf_rf_{'numTrees': 20}...


Running tfidf_rf_{'numTrees': 50}...


Running bigram_tfidf_lr_{'maxIter': 20}...


...

Running bigram_tfidf_rf_{'numTrees': 20}...


26/05/02 04:23:45 WARN DAGScheduler: Broadcasting large task binary with size 1822.7 KiB
26/05/02 04:23:46 WARN MemoryStore: Not enough space to cache rdd_9960_0 in memory! (computed 343.5 MiB so far)
26/05/02 04:23:46 WARN BlockManager: Persisting block rdd_9960_0 to disk instead.
26/05/02 04:23:47 WARN MemoryStore: Not enough space to cache rdd_9960_0 in memory! (computed 343.5 MiB so far)
26/05/02 04:23:47 WARN DAGScheduler: Broadcasting large task binary with size 1833.2 KiB
26/05/02 04:23:47 WARN MemoryStore: Not enough space to cache rdd_9960_0 in memory! (computed 343.5 MiB so far)
26/05/02 04:23:48 WARN DAGScheduler: Broadcasting large task binary with size 1845.3 KiB
26/05/02 04:23:48 WARN MemoryStore: Not enough space to cache rdd_9960_0 in memory! (computed 343.5 MiB so far)
26/05/02 04:23:48 WARN DAGScheduler: Broadcasting large task binary with size 1854.0 KiB
26/05/02 04:23:49 WARN MemoryStore: Not enough space to cache rdd_9960_0 in memory! (computed 343.5 MiB so far)
26

Running bigram_tfidf_rf_{'numTrees': 50}...


26/05/02 04:23:50 WARN DAGScheduler: Broadcasting large task binary with size 1164.4 KiB
26/05/02 04:23:54 WARN DAGScheduler: Broadcasting large task binary with size 1849.2 KiB
26/05/02 04:23:55 WARN MemoryStore: Not enough space to cache rdd_10045_0 in memory! (computed 343.8 MiB so far)
26/05/02 04:23:55 WARN BlockManager: Persisting block rdd_10045_0 to disk instead.
26/05/02 04:23:56 WARN MemoryStore: Not enough space to cache rdd_10045_0 in memory! (computed 343.8 MiB so far)
26/05/02 04:23:56 WARN DAGScheduler: Broadcasting large task binary with size 1879.9 KiB
26/05/02 04:23:56 WARN MemoryStore: Not enough space to cache rdd_10045_0 in memory! (computed 343.8 MiB so far)
26/05/02 04:23:57 WARN DAGScheduler: Broadcasting large task binary with size 1903.1 KiB
26/05/02 04:23:57 WARN MemoryStore: Not enough space to cache rdd_10045_0 in memory! (computed 343.8 MiB so far)
26/05/02 04:23:57 WARN DAGScheduler: Broadcasting large task binary with size 1923.7 KiB
26/05/02 04:23:58 WA

Running word2vec_lr_{'maxIter': 20}...
Running word2vec_lr_{'maxIter': 50}...
Running word2vec_dt_{'maxDepth': 5}...
Running word2vec_dt_{'maxDepth': 10}...
Running word2vec_rf_{'numTrees': 20}...
Running word2vec_rf_{'numTrees': 50}...


In [78]:
best_model = max(all_results.items(), key=lambda x: x[1]["f1"])

print("Best configuration:")
print(best_model[0])
print(best_model[1])

Best configuration:
tfidf_dt_{'maxDepth': 10}
{'accuracy': 0.7081468218442256, 'f1': 0.6619221702227189, 'weightedPrecision': 0.6837594132017342, 'weightedRecall': 0.7081468218442257}


In [79]:
import pandas as pd

results_table = []

for name, metrics in all_results.items():
    row = {"experiment": name}
    row.update(metrics)
    results_table.append(row)

results_df = pd.DataFrame(results_table)
results_df.sort_values(by="f1", ascending=False)

,experiment,accuracy,f1,weightedPrecision,weightedRecall
3,tfidf_dt_{'maxDepth': 10},0.708147,0.661922,0.683759,0.708147
2,tfidf_dt_{'maxDepth': 5},0.691137,0.638574,0.679942,0.691137
13,word2vec_lr_{'maxIter': 50},0.648165,0.606488,0.592757,0.648165
0,tfidf_lr_{'maxIter': 20},0.602507,0.605680,0.609046,0.602507
12,word2vec_lr_{'maxIter': 20},0.647269,0.601838,0.607220,0.647269
1,tfidf_lr_{'maxIter': 50},0.595345,0.599420,0.603785,0.595345
17,word2vec_rf_{'numTrees': 50},0.660698,0.595649,0.557720,0.660698
16,word2vec_rf_{'numTrees': 20},0.658013,0.593009,0.555999,0.658013
14,word2vec_dt_{'maxDepth': 5},0.646374,0.592220,0.637787,0.646374
15,word2vec_dt_{'maxDepth': 10},0.601611,0.588675,0.579729,0.601611


### Interpretacija rezultata

Na osnovu sprovedenih eksperimenata može se zaključiti da je najbolji model za analizu sentimenta na Financial Sentiment Analysis skupu podataka bio **Decision Tree** sa **TF-IDF** reprezentacijom teksta i hiperparametrom **maxDepth = 10**.

Ovaj model je ostvario:

- Accuracy: 0.708
- F1 score: 0.662
- Weighted Precision: 0.684
- Weighted Recall: 0.708

Rezultati pokazuju da je **TF-IDF** reprezentacija bila efikasnija od **Word2Vec** i **TF-IDF sa bigramima**, što ukazuje da su pojedinačne reči bile dovoljno informativne za klasifikaciju sentimenta finansijskih vesti.

Modeli bazirani na **Word2Vec** ostvarili su solidne rezultate, ali nešto slabije od TF-IDF pristupa, verovatno zbog gubitka specifičnih informacija važnih za sentiment.

**TF-IDF sa bigramima** dao je slabije performanse, što može biti posledica povećane dimenzionalnosti i relativno malog broja uzoraka.

Najslabije rezultate dao je **Random Forest**, posebno na sparse TF-IDF reprezentacijama, što je očekivano zbog velike dimenzionalnosti ulaznog prostora.

### Zaključak

Na osnovu dobijenih rezultata preporučuje se upotreba **TF-IDF reprezentacije teksta** u kombinaciji sa **Decision Tree klasifikatorom**, jer je ovaj pristup dao najbolje rezultate na posmatranom skupu podataka.

Ovakav pristup je pogodan za praktične primene analize sentimenta u finansijskom domenu, gde su ključne reči i termini od velike važnosti za određivanje sentimenta.